# NYC Building Energy Benchmarking - Part G: Regression Analysis
## TY B.Tech Integrated Data Warehouse, OLAP and Data Mining Project
**Domain:** Building Energy and Sustainability Analytics (NYC Local Law 84)

### Objectives:
1. Ingest from analytical mart `data/mart/building_performance_mart.csv` (architecturally generated from warehouse)
2. Target Variable: `SiteEUI` (Site Energy Use Intensity)
3. Predictors: `GrossFloorArea`, `PrimaryPropertyType`, `Borough`, `ReportingYear`
4. Strict Chronological Split to prevent data leakage (earlier years for train, final year for test)
5. Fit Simple Linear Regression & Multiple Linear Regression
6. Evaluate using MAE, MSE, RMSE, R-squared
7. Analyze residuals, multicollinearity (VIF), and heteroscedasticity


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
MART_PATH = '../data/mart/building_performance_mart.csv'

df_mart = pd.read_csv(MART_PATH)
print(f"Ingested analytical mart: {len(df_mart):,} records × {len(df_mart.columns)} columns")


Ingested analytical mart: 7,370 records × 24 columns


### 1. Feature Preprocessing and Chronological Train/Test Partition

In [2]:
features = ['GrossFloorArea', 'PrimaryPropertyType', 'Borough', 'Year']
target = 'SiteEUI'

df_reg = df_mart.dropna(subset=features + [target]).copy()

train_mask = df_reg['Year'] < 2024
test_mask = df_reg['Year'] == 2024

X_raw = df_reg[features]
y = df_reg[target].values

X_encoded = pd.get_dummies(X_raw.drop(columns=['Year']), drop_first=True)
X_encoded['GFA_Scaled'] = (X_encoded['GrossFloorArea'] - X_encoded['GrossFloorArea'].mean()) / X_encoded['GrossFloorArea'].std()
X_encoded = X_encoded.drop(columns=['GrossFloorArea'])
X_encoded['Intercept'] = 1.0

X_train = X_encoded[train_mask].values.astype(float)
y_train = y[train_mask].astype(float)
X_test = X_encoded[test_mask].values.astype(float)
y_test = y[test_mask].astype(float)

print(f"Training Set (2022-2023): {len(y_train):,} rows")
print(f"Testing Set (2024):        {len(y_test):,} rows")


Training Set (2022-2023): 4,906 rows
Testing Set (2024):        2,464 rows


### 2. Simple Linear Regression (SiteEUI ~ GrossFloorArea)

In [3]:
gfa_col_idx = X_encoded.columns.get_loc('GFA_Scaled')
X_simple_train = np.column_stack([np.ones(len(y_train)), X_train[:, gfa_col_idx]])
X_simple_test = np.column_stack([np.ones(len(y_test)), X_test[:, gfa_col_idx]])

beta_simple = np.linalg.pinv(X_simple_train.T @ X_simple_train) @ (X_simple_train.T @ y_train)
y_pred_simple = X_simple_test @ beta_simple

mae_simple = np.mean(np.abs(y_test - y_pred_simple))
rmse_simple = np.sqrt(np.mean((y_test - y_pred_simple)**2))
ss_tot = np.sum((y_test - np.mean(y_test))**2)
ss_res_simple = np.sum((y_test - y_pred_simple)**2)
r2_simple = 1 - (ss_res_simple / ss_tot)

print("=== Simple Linear Regression Results ===")
print(f"Intercept: {beta_simple[0]:.4f}")
print(f"Slope (GFA_Scaled): {beta_simple[1]:.4f}")
print(f"MAE: {mae_simple:.2f}")
print(f"RMSE: {rmse_simple:.2f}")
print(f"R²: {r2_simple:.4f}")


=== Simple Linear Regression Results ===
Intercept: 83.5983
Slope (GFA_Scaled): 0.3358
MAE: 21.84
RMSE: 36.65
R²: -0.0031


### 3. Multiple Linear Regression

In [4]:
beta_multi = np.linalg.pinv(X_train.T @ X_train) @ (X_train.T @ y_train)
y_pred_multi = X_test @ beta_multi

mae_multi = np.mean(np.abs(y_test - y_pred_multi))
mse_multi = np.mean((y_test - y_pred_multi)**2)
rmse_multi = np.sqrt(mse_multi)
ss_res_multi = np.sum((y_test - y_pred_multi)**2)
r2_multi = 1 - (ss_res_multi / ss_tot)

print("=== Multiple Linear Regression Results ===")
print(f"MAE:  {mae_multi:.2f}")
print(f"MSE:  {mse_multi:.2f}")
print(f"RMSE: {rmse_multi:.2f}")
print(f"R²:   {r2_multi:.4f}")

coef_df = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Coefficient': beta_multi
}).sort_values(by='Coefficient', ascending=False)

print('Top 8 Feature Coefficients:')
print(coef_df.head(8).to_string(index=False))


=== Multiple Linear Regression Results ===
MAE:  13.68
MSE:  577.18
RMSE: 24.02
R²:   0.5689
Top 8 Feature Coefficients:
                                                  Feature  Coefficient
PrimaryPropertyType_Hospital (General Medical & Surgical)   131.381879
            PrimaryPropertyType_Supermarket/Grocery Store   112.903477
                                                Intercept   106.377407
                                PrimaryPropertyType_Hotel     6.748453
                                           Borough_Queens     1.491616
                                    Borough_Staten Island     0.952300
                                               GFA_Scaled    -0.132113
                                        Borough_Manhattan    -0.624822


### 4. Residual and Diagnostics Analysis

In [5]:
residuals = y_test - y_pred_multi

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_pred_multi, residuals, alpha=0.3, color='navy')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals vs Fitted Values')
axes[0].set_xlabel('Predicted Site EUI')
axes[0].set_ylabel('Residual')
axes[0].grid(True, alpha=0.3)

axes[1].hist(residuals, bins=35, color='purple', edgecolor='black', alpha=0.7)
axes[1].set_title('Residuals Distribution')
axes[1].set_xlabel('Residual Error')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean Residual: {np.mean(residuals):.4f}")
print(f"Residual Std Dev: {np.std(residuals):.4f}")


Mean Residual: -2.0227
Residual Std Dev: 23.9393
